# Context-Aware HiRISE Reconstruction — Inspection

Loads a saved checkpoint, samples random local/context patch pairs from the index, and visualises the masked reconstruction side by side with the originals.

In [12]:
# ── CUDA environment setup — must be the FIRST cell run in a fresh kernel ─────
#
# libcudnn.so.9 is not on LD_LIBRARY_PATH because the Jupyter kernel starts
# without the cluster module environment.  We load it via ctypes so that
# PyTorch's own dlopen call finds it already in the process symbol table.
#
# IMPORTANT: if you see any torch ImportError or AttributeError, do NOT re-run
# this cell — restart the kernel instead.  C extensions cannot be reloaded in
# the same process once they have been partially initialised.
#
import ctypes
import os
import sys

if 'torch' in sys.modules:
    raise RuntimeError(
        "torch is already imported — this cell must run before any import of torch.\n"
        "Restart the kernel (Kernel → Restart), then re-run from this cell."
    )

_CUDNN_PATH = '/is/software/nvidia/cudnn-9.10.2/lib/libcudnn.so.9'
if os.path.exists(_CUDNN_PATH):
    ctypes.CDLL(_CUDNN_PATH, mode=ctypes.RTLD_GLOBAL)
    print(f'Preloaded {_CUDNN_PATH}')
else:
    print('cuDNN not found — torch will try the system linker.')

Preloaded /is/software/nvidia/cudnn-9.10.2/lib/libcudnn.so.9


In [13]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader


def _find_vision_backend() -> Path:
    """Walk upward from cwd until we find vision_backend/model/model.py."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'model' / 'model.py').exists():
            return candidate          # cwd is already inside vision_backend/
        if (candidate / 'vision_backend' / 'model' / 'model.py').exists():
            return candidate / 'vision_backend'
    raise RuntimeError(
        f"Cannot locate vision_backend/ from {start}. "
        "Run Jupyter from the repo root or from vision_backend/."
    )

VISION_BACKEND_DIR = _find_vision_backend()
PROJECT_ROOT = VISION_BACKEND_DIR.parent

if str(VISION_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(VISION_BACKEND_DIR))

print('vision_backend:', VISION_BACKEND_DIR)
print('project root  :', PROJECT_ROOT)

from context_reconstruction import (
    HiRISEContextPatchDataset,
    group_records_by_source,
    load_patch_records,
)
from training.builders import build_context_pretrainer

RuntimeError: function '_has_torch_function' already has a docstring

## Configuration

Edit the paths and model hyperparameters here. `MODEL_CONFIG` must match the architecture that was used during training.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
INDEX_PATH      = PROJECT_ROOT / 'data' / 'patch_index.csv'
CHECKPOINT_PATH = PROJECT_ROOT / 'results' / 'stage1_training' / 'stage1_1ep_best_model.pt'

# ── Data ─────────────────────────────────────────────────────────────────────
LOCAL_INPUT_SIZE   = 256
CONTEXT_INPUT_SIZE = 256
BATCH_SIZE         = 8
DATASET_BACKEND    = 'auto'   # 'auto' | 'offline_paired_context' | 'offline_shared_context' | 'online_jp2'

# ── Model (must match the checkpoint) ────────────────────────────────────────
MODEL_CONFIG = {
    'in_channels':           1,
    'local_base_channels':   48,
    'context_base_channels': 24,
    'context_dim':           256,
    'decoder_channels':      256,
    'mask_patch_size':       16,
    'mask_ratio':            0.6,
    'window_size':           8,
    'swin_depths':           (2, 2, 2),
    'swin_num_heads':        (4, 8, 16),
    'use_stage32':           False,
    'loss_type':             'l1',
    'drop_path':             0.0,
}

# ── Visualisation ─────────────────────────────────────────────────────────────
NUM_PAIRS = 6   # random context pairs to display
SEED      = 0

## Device

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('device:', device)

## Load Patch Index

In [ ]:
records = load_patch_records(INDEX_PATH)
sources = group_records_by_source(records)

print(f'Total patches : {len(records)}')
print(f'Source images : {len(sources)}')
for src, recs in sorted(sources.items()):
    print(f'  {Path(src).name}  ({len(recs)} patches)')

## Build Model and Load Checkpoint

In [ ]:
model = build_context_pretrainer(MODEL_CONFIG).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)

# Support checkpoints saved with different key conventions
for key in ('model_state', 'state_dict', 'model'):
    if key in checkpoint and isinstance(checkpoint[key], dict):
        state_dict = checkpoint[key]
        break
else:
    state_dict = checkpoint  # bare state dict

missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f'Missing keys ({len(missing)}):', missing[:5], '...' if len(missing) > 5 else '')
if unexpected:
    print(f'Unexpected keys ({len(unexpected)}):', unexpected[:5], '...' if len(unexpected) > 5 else '')
if not missing and not unexpected:
    print('Checkpoint loaded cleanly.')

# Log epoch / step info if present
for info_key in ('epoch', 'step', 'best_val_loss', 'val_loss'):
    if info_key in checkpoint:
        print(f'  {info_key}: {checkpoint[info_key]}')

model.eval();

## Random Context Pairs

Samples `NUM_PAIRS` random patches from the full index and shows the context window alongside the local patch — no model involved.

In [ ]:
def _stretch(arr: np.ndarray, lo: float = 1.0, hi: float = 99.0) -> np.ndarray:
    """Percentile-stretch to [0, 1] for display."""
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 3 and arr.shape[0] <= 4:
        arr = np.moveaxis(arr, 0, -1)
    if arr.ndim == 3 and arr.shape[2] == 1:
        arr = arr[:, :, 0]
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr)
    vlo, vhi = float(np.percentile(finite, lo)), float(np.percentile(finite, hi))
    if vhi <= vlo:
        vlo, vhi = float(finite.min()), float(finite.max())
    if vhi <= vlo:
        return np.zeros_like(arr)
    return np.clip((arr - vlo) / (vhi - vlo), 0.0, 1.0)


def show_tensor(ax, t, title: str = '', stretch: bool = True) -> None:
    img = t.squeeze().detach().cpu().numpy() if isinstance(t, torch.Tensor) else np.asarray(t)
    ax.imshow(_stretch(img) if stretch else np.clip(img, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis('off')


rng = random.Random(SEED)
sampled = rng.sample(records, min(NUM_PAIRS, len(records)))

dataset = HiRISEContextPatchDataset(
    sampled,
    dataset_backend=DATASET_BACKEND,
    local_input_size=LOCAL_INPUT_SIZE,
    context_input_size=CONTEXT_INPUT_SIZE,
)

fig, axes = plt.subplots(len(sampled), 2, figsize=(6, 3 * len(sampled)))
if len(sampled) == 1:
    axes = axes[None, :]

for row_idx, record in enumerate(sampled):
    item = dataset[row_idx]
    show_tensor(axes[row_idx, 0], item['context'], 'Context window')
    show_tensor(axes[row_idx, 1], item['local'],   'Local patch')
    axes[row_idx, 0].set_ylabel(
        f"r{record.row} c{record.col}",
        fontsize=7, rotation=0, labelpad=50, va='center'
    )

fig.suptitle('Random local / context pairs (no model)', fontsize=11)
plt.tight_layout()
plt.show()

## Reconstruction

Runs the loaded model on the same randomly sampled patches and displays:
context input · original local · masked input · reconstruction · mask

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

COLS = ['Context', 'Original', 'Masked input', 'Reconstruction', 'Mask']

all_context = []
all_original = []
all_masked = []
all_recon = []
all_mask = []

model.eval()
with torch.no_grad():
    for batch in loader:
        local_t   = batch['local'].to(device).float()
        context_t = batch['context'].to(device).float()
        outputs   = model(local_t, context_t)

        all_context  += list(context_t.cpu())
        all_original += list(local_t.cpu())
        all_masked   += list(outputs['masked_input'].cpu())
        all_recon    += list(outputs['reconstruction'].cpu())
        all_mask     += list(outputs['mask'].cpu())

n = len(all_original)
fig, axes = plt.subplots(n, len(COLS), figsize=(3 * len(COLS), 3 * n))
if n == 1:
    axes = axes[None, :]

for col_idx, title in enumerate(COLS):
    axes[0, col_idx].set_title(title, fontsize=9)

for row_idx in range(n):
    show_tensor(axes[row_idx, 0], all_context[row_idx],  stretch=True)
    show_tensor(axes[row_idx, 1], all_original[row_idx], stretch=True)
    show_tensor(axes[row_idx, 2], all_masked[row_idx],   stretch=True)
    show_tensor(axes[row_idx, 3], all_recon[row_idx],    stretch=True)
    show_tensor(axes[row_idx, 4], all_mask[row_idx],     stretch=False)

fig.suptitle(f'Masked reconstruction — {Path(CHECKPOINT_PATH).name}', fontsize=11)
plt.tight_layout()
plt.show()

## Re-sample and Re-run

Change `SEED` and `NUM_PAIRS` in the **Configuration** cell and re-run from there, or run the cell below to pick a new random seed on the fly.

In [ ]:
import time

new_seed = int(time.time()) % 10_000
print(f'New seed: {new_seed}  — re-run the two visualisation cells above after updating SEED, or just run this cell and the next two.')

rng2 = random.Random(new_seed)
sampled2 = rng2.sample(records, min(NUM_PAIRS, len(records)))

dataset2 = HiRISEContextPatchDataset(
    sampled2,
    dataset_backend=DATASET_BACKEND,
    local_input_size=LOCAL_INPUT_SIZE,
    context_input_size=CONTEXT_INPUT_SIZE,
)

# ── pairs ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(sampled2), 2, figsize=(6, 3 * len(sampled2)))
if len(sampled2) == 1:
    axes = axes[None, :]
for row_idx, record in enumerate(sampled2):
    item = dataset2[row_idx]
    show_tensor(axes[row_idx, 0], item['context'], 'Context window')
    show_tensor(axes[row_idx, 1], item['local'],   'Local patch')
    axes[row_idx, 0].set_ylabel(f"r{record.row} c{record.col}", fontsize=7, rotation=0, labelpad=50, va='center')
fig.suptitle(f'Random pairs (seed={new_seed})', fontsize=11)
plt.tight_layout()
plt.show()

# ── reconstruction ─────────────────────────────────────────────────────────
loader2 = DataLoader(dataset2, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
c2, o2, m2, r2, mk2 = [], [], [], [], []
model.eval()
with torch.no_grad():
    for batch in loader2:
        lt = batch['local'].to(device).float()
        ct = batch['context'].to(device).float()
        out = model(lt, ct)
        c2  += list(ct.cpu());  o2 += list(lt.cpu())
        m2  += list(out['masked_input'].cpu())
        r2  += list(out['reconstruction'].cpu())
        mk2 += list(out['mask'].cpu())

n2 = len(o2)
fig2, axes2 = plt.subplots(n2, len(COLS), figsize=(3 * len(COLS), 3 * n2))
if n2 == 1:
    axes2 = axes2[None, :]
for ci, title in enumerate(COLS):
    axes2[0, ci].set_title(title, fontsize=9)
for ri in range(n2):
    show_tensor(axes2[ri, 0], c2[ri],  stretch=True)
    show_tensor(axes2[ri, 1], o2[ri],  stretch=True)
    show_tensor(axes2[ri, 2], m2[ri],  stretch=True)
    show_tensor(axes2[ri, 3], r2[ri],  stretch=True)
    show_tensor(axes2[ri, 4], mk2[ri], stretch=False)
fig2.suptitle(f'Reconstruction (seed={new_seed})', fontsize=11)
plt.tight_layout()
plt.show()